<a href="https://colab.research.google.com/github/Nishan-vxm/AI-Business-Copilot-/blob/main/AI_business_copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [ ]:
import os

files = os.listdir(path)

for file in files:
    print(file)

olist_customers_dataset.csv
olist_sellers_dataset.csv
olist_order_reviews_dataset.csv
olist_order_items_dataset.csv
olist_products_dataset.csv
olist_geolocation_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_payments_dataset.csv


In [ ]:
import pandas as pd
import os
import glob

csv_files = glob.glob(os.path.join(path, "*.csv"))

dataframes = {}

for file in csv_files:
    name = os.path.basename(file).replace(".csv", "")
    dataframes[name] = pd.read_csv(file)
    print(name, dataframes[name].shape)

print("\nTotal DataFrames:", len(dataframes))

olist_customers_dataset (99441, 5)
olist_sellers_dataset (3095, 4)
olist_order_reviews_dataset (99224, 7)
olist_order_items_dataset (112650, 7)
olist_products_dataset (32951, 9)
olist_geolocation_dataset (1000163, 5)
product_category_name_translation (71, 2)
olist_orders_dataset (99441, 8)
olist_order_payments_dataset (103886, 5)

Total DataFrames: 9


In [ ]:

print(dataframes.keys())

dict_keys(['olist_customers_dataset', 'olist_sellers_dataset', 'olist_order_reviews_dataset', 'olist_order_items_dataset', 'olist_products_dataset', 'olist_geolocation_dataset', 'product_category_name_translation', 'olist_orders_dataset', 'olist_order_payments_dataset'])


In [ ]:

df = dataframes["olist_orders_dataset"]

display(df.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
df.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object


In [ ]:
import sqlite3

db_path = "/content/olist_business.db"

conn = sqlite3.connect(db_path)

for name, dataframe in dataframes.items():
    dataframe.to_sql(
        name,
        conn,
        if_exists="replace",
        index=False
    )

print("All DataFrames loaded into SQLite!")

All DataFrames loaded into SQLite!


In [ ]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

display(tables)

,name
0,olist_customers_dataset
1,olist_geolocation_dataset
2,olist_order_items_dataset
3,olist_order_payments_dataset
4,olist_order_reviews_dataset
5,olist_orders_dataset
6,olist_products_dataset
7,olist_sellers_dataset
8,product_category_name_translation


In [ ]:
query = """
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM olist_orders_dataset
GROUP BY order_status
ORDER BY total_orders DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/content/olist_business.db")

for name, dataframe in dataframes.items():
    dataframe.to_sql(
        name,
        conn,
        if_exists="replace",
        index=False
    )

print("Tables created:", len(dataframes))

Tables created: 9


In [ ]:
pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

,name
0,olist_customers_dataset
1,olist_geolocation_dataset
2,olist_order_items_dataset
3,olist_order_payments_dataset
4,olist_order_reviews_dataset
5,olist_orders_dataset
6,olist_products_dataset
7,olist_sellers_dataset
8,product_category_name_translation


In [ ]:
import pandas as pd

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn)

for table in tables["name"]:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table}")
    print("=" * 60)

    schema = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        conn
    )

    print(schema[["name", "type", "notnull", "pk"]].to_string(index=False))


TABLE: olist_customers_dataset
                    name    type  notnull  pk
             customer_id    TEXT        0   0
      customer_unique_id    TEXT        0   0
customer_zip_code_prefix INTEGER        0   0
           customer_city    TEXT        0   0
          customer_state    TEXT        0   0

TABLE: olist_geolocation_dataset
                       name    type  notnull  pk
geolocation_zip_code_prefix INTEGER        0   0
            geolocation_lat    REAL        0   0
            geolocation_lng    REAL        0   0
           geolocation_city    TEXT        0   0
          geolocation_state    TEXT        0   0

TABLE: olist_order_items_dataset
               name    type  notnull  pk
           order_id    TEXT        0   0
      order_item_id INTEGER        0   0
         product_id    TEXT        0   0
          seller_id    TEXT        0   0
shipping_limit_date    TEXT        0   0
              price    REAL        0   0
      freight_value    REAL        0   0

T

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "/content/olist_business.db"

def run_sql(query):
    """
    Execute read-only SQL queries and return results as a DataFrame.
    """
    connection = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True
    )

    try:
        connection.execute("PRAGMA query_only = ON")

        cursor = connection.execute(query)

        if cursor.description is None:
            raise ValueError("Only queries returning results are allowed.")

        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchmany(1000)

        return pd.DataFrame(rows, columns=columns)

    except sqlite3.Error as e:
        print("SQL Error:", e)
        return None

    finally:
        connection.close()

In [ ]:
result = run_sql("""
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM olist_orders_dataset
GROUP BY order_status
ORDER BY total_orders DESC
""")

display(result)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [ ]:
def analyze_dataframe(df, operation, column=None, top_n=10):
    """
    Basic reusable Pandas analysis tool.
    Supported operations:
    head, describe, missing, value_counts, group_mean
    """

    if operation == "head":
        return df.head(top_n)

    elif operation == "describe":
        return df.describe(include="all").T

    elif operation == "missing":
        return df.isnull().sum().to_frame("missing_values")

    elif operation == "value_counts":
        if column not in df.columns:
            raise ValueError(f"Unknown column: {column}")

        return df[column].value_counts().head(top_n).to_frame("count")

    elif operation == "group_mean":
        if column not in df.columns:
            raise ValueError(f"Unknown column: {column}")

        numeric_columns = df.select_dtypes(include="number").columns

        if len(numeric_columns) == 0:
            raise ValueError("No numeric columns available.")

        return df.groupby(column)[numeric_columns].mean()

    else:
        raise ValueError("Unsupported operation.")

In [ ]:
result = analyze_dataframe(
    df,
    operation="value_counts",
    column="order_status",
    top_n=10
)

display(result)

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [ ]:
TOOLS = {
    "run_sql": {
        "description": "Execute a read-only SQL query on the business database.",
        "function": run_sql,
        "input_schema": {
            "query": "string"
        }
    },
    "analyze_dataframe": {
        "description": "Analyze a Pandas DataFrame using supported operations.",
        "function": analyze_dataframe,
        "input_schema": {
            "df": "DataFrame",
            "operation": "head | describe | missing | value_counts | group_mean",
            "column": "optional string",
            "top_n": "optional integer"
        }
    }
}

for name, tool in TOOLS.items():
    print(f"{name}: {tool['description']}")

run_sql: Execute a read-only SQL query on the business database.
analyze_dataframe: Analyze a Pandas DataFrame using supported operations.


In [ ]:
print("Available tools:", list(TOOLS.keys()))

test_result = TOOLS["run_sql"]["function"]("""
SELECT COUNT(*) AS total_orders
FROM olist_orders_dataset
""")

display(test_result)

Available tools: ['run_sql', 'analyze_dataframe']


,total_orders
0,99441


In [ ]:
def get_schema():
    connection = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True
    )

    try:
        tables = connection.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
        """).fetchall()

        schema_info = {}

        for (table_name,) in tables:
            columns = connection.execute(
                f'PRAGMA table_info("{table_name}")'
            ).fetchall()

            schema_info[table_name] = [
                {
                    "column": col[1],
                    "type": col[2]
                }
                for col in columns
            ]

        return schema_info

    finally:
        connection.close()


schema = get_schema()

print("Total tables:", len(schema))

for table, columns in schema.items():
    print(f"\n{table}:")
    print([col["column"] for col in columns])

Total tables: 9

olist_customers_dataset:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_geolocation_dataset:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

olist_order_items_dataset:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_order_payments_dataset:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

olist_order_reviews_dataset:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

olist_orders_dataset:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_products_dataset:
['product_id', 'product_category_name', 'product_

In [ ]:
def generate_sql(question):
    q = question.lower().strip()

    if "total orders" in q or "how many orders" in q:
        return """
        SELECT COUNT(*) AS total_orders
        FROM olist_orders_dataset
        """

    elif "order status" in q:
        return """
        SELECT order_status, COUNT(*) AS total_orders
        FROM olist_orders_dataset
        GROUP BY order_status
        ORDER BY total_orders DESC
        """

    elif "top customers" in q:
        return """
        SELECT customer_id, COUNT(*) AS total_orders
        FROM olist_orders_dataset
        GROUP BY customer_id
        ORDER BY total_orders DESC
        LIMIT 10
        """

    else:
        return None

In [ ]:
question = "How many orders are there?"

sql = generate_sql(question)

print("Generated SQL:")
print(sql)

if sql:
    result = run_sql(sql)
    display(result)
else:
    print("Question not supported by baseline.")

Generated SQL:

        SELECT COUNT(*) AS total_orders
        FROM olist_orders_dataset
        


,total_orders
0,99441


In [ ]:
import re

def validate_sql(query):
    if not isinstance(query, str) or not query.strip():
        return False, "Empty query"

    q = query.strip()

    # Only SELECT queries allowed
    if not re.match(r"(?is)^SELECT\b", q):
        return False, "Only SELECT queries are allowed"

    # Reject multiple statements
    cleaned = q.rstrip().rstrip(";")
    if ";" in cleaned:
        return False, "Multiple SQL statements are not allowed"

    # Reject common write/administrative keywords
    if re.search(
        r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|ATTACH|DETACH|"
        r"REPLACE|PRAGMA|VACUUM|REINDEX|TRIGGER)\b",
        q,
        re.IGNORECASE
    ):
        return False, "Forbidden SQL keyword detected"

    return True, "Query passed basic checks"

In [ ]:
tests = [
    "SELECT COUNT(*) FROM olist_orders_dataset",
    "DELETE FROM olist_orders_dataset",
    "SELECT * FROM olist_orders_dataset; DROP TABLE orders"
]

for query in tests:
    valid, message = validate_sql(query)
    print(query, "=>", valid, "|", message)

SELECT COUNT(*) FROM olist_orders_dataset => True | Query passed basic checks
DELETE FROM olist_orders_dataset => False | Only SELECT queries are allowed
SELECT * FROM olist_orders_dataset; DROP TABLE orders => False | Multiple SQL statements are not allowed


In [ ]:
def business_copilot_baseline(question):
    print("User question:", question)


    sql = generate_sql(question)

    if not sql:
        return {
            "status": "unsupported",
            "message": "This question is not supported by the current baseline."
        }

    print("\nGenerated SQL:")
    print(sql)


    valid, message = validate_sql(sql)

    if not valid:
        return {
            "status": "rejected",
            "message": message
        }


    result = run_sql(sql)

    if result is None:
        return {
            "status": "error",
            "message": "SQL execution failed."
        }


    return {
        "status": "success",
        "sql": sql,
        "result": result
    }

In [ ]:
response = business_copilot_baseline(
    "How many orders are there?"
)

print("Status:", response["status"])
print("SQL:", response.get("sql"))

if response["status"] == "success":
    display(response["result"])
else:
    print(response["message"])

User question: How many orders are there?

Generated SQL:

        SELECT COUNT(*) AS total_orders
        FROM olist_orders_dataset
        
Status: success
SQL: 
        SELECT COUNT(*) AS total_orders
        FROM olist_orders_dataset
        


,total_orders
0,99441


In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "/content/olist_business.db"

ALLOWED_TABLES = set(schema.keys())

def safe_run_sql(query, max_rows=1000, timeout_ms=3000):
    valid, message = validate_sql(query)

    if not valid:
        raise ValueError(f"Query rejected: {message}")

    conn = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True,
        timeout=3
    )

    try:
        conn.execute("PRAGMA query_only = ON")
        conn.set_authorizer(
            lambda action, arg1, arg2, db, trigger:
                sqlite3.SQLITE_OK
                if action == sqlite3.SQLITE_SELECT
                or action == sqlite3.SQLITE_READ
                or action == sqlite3.SQLITE_FUNCTION
                or action == sqlite3.SQLITE_RECURSIVE
                else sqlite3.SQLITE_DENY
        )

        conn.set_progress_handler(
            lambda: 1, timeout_ms
        )

        cursor = conn.execute(query)

        if cursor.description is None:
            raise ValueError("Query must return tabular results.")

        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchmany(max_rows)

        return pd.DataFrame(rows, columns=columns)

    finally:
        conn.close()

In [ ]:
import sqlite3
import pandas as pd
import time

DB_PATH = "/content/olist_business.db"

def safe_run_sql(query, max_rows=1000, timeout_ms=3000):
    valid, message = validate_sql(query)

    if not valid:
        raise ValueError(f"Query rejected: {message}")

    conn = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True,
        timeout=3
    )

    start_time = time.perf_counter()

    try:
        conn.execute("PRAGMA query_only = ON")


        allowed_actions = {
            sqlite3.SQLITE_SELECT,
            sqlite3.SQLITE_READ,
            sqlite3.SQLITE_FUNCTION,
            sqlite3.SQLITE_RECURSIVE
        }

        def authorize(action, arg1, arg2, db_name, trigger):
            if action in allowed_actions:
                return sqlite3.SQLITE_OK
            return sqlite3.SQLITE_DENY

        conn.set_authorizer(authorize)


        def check_timeout():
            elapsed = time.perf_counter() - start_time
            return 1 if elapsed > timeout_ms / 1000 else 0

        conn.set_progress_handler(check_timeout, 1000)

        cursor = conn.execute(query)

        if cursor.description is None:
            raise ValueError("Query must return results.")

        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchmany(max_rows)

        return pd.DataFrame(rows, columns=columns)

    except sqlite3.Error as e:
        if "interrupted" in str(e).lower():
            raise TimeoutError("Query exceeded the time limit.") from e
        raise

    finally:
        conn.close()

In [ ]:
result = safe_run_sql("""
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM olist_orders_dataset
GROUP BY order_status
ORDER BY total_orders DESC
""")

display(result)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [ ]:
tests = [
    (
        "Valid SELECT",
        "SELECT COUNT(*) AS total_orders FROM olist_orders_dataset"
    ),
    (
        "Write query",
        "DELETE FROM olist_orders_dataset"
    ),
    (
        "Multiple statements",
        "SELECT 1; DROP TABLE olist_orders_dataset"
    ),
]

for name, query in tests:
    print(f"\nTest: {name}")

    try:
        result = safe_run_sql(query)
        print("Result: ALLOWED")
        display(result)
    except Exception as e:
        print("Result: BLOCKED")
        print(type(e).__name__, str(e))


Test: Valid SELECT
Result: ALLOWED


,total_orders
0,99441



Test: Write query
Result: BLOCKED
ValueError Query rejected: Only SELECT queries are allowed

Test: Multiple statements
Result: BLOCKED
ValueError Query rejected: Multiple SQL statements are not allowed


In [ ]:
import sqlite3


ALLOWED_TABLES = set(schema.keys())

def check_table_access(query):
    conn = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True
    )

    accessed_tables = set()

    def authorize(action, arg1, arg2, db_name, trigger):
        if action == sqlite3.SQLITE_READ:
            if arg1:
                accessed_tables.add(arg1)

            if arg1 and arg1 not in ALLOWED_TABLES:
                return sqlite3.SQLITE_DENY

        if action in {
            sqlite3.SQLITE_SELECT,
            sqlite3.SQLITE_FUNCTION,
            sqlite3.SQLITE_RECURSIVE
        }:
            return sqlite3.SQLITE_OK

        return sqlite3.SQLITE_DENY

    try:
        conn.set_authorizer(authorize)
        conn.execute("PRAGMA query_only = ON")
        conn.execute("EXPLAIN QUERY PLAN " + query).fetchall()

        return {
            "allowed": True,
            "tables": sorted(accessed_tables)
        }

    except sqlite3.Error as e:
        return {
            "allowed": False,
            "error": str(e)
        }

    finally:
        conn.close()

In [ ]:
test_query = """
SELECT order_status, COUNT(*)
FROM olist_orders_dataset
GROUP BY order_status
"""

print(check_table_access(test_query))

{'allowed': False, 'error': 'not authorized'}


In [ ]:
APPROVED_TABLES = {
    "olist_orders_dataset",
    "olist_customers_dataset",
    "olist_order_items_dataset",
    "olist_products_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_sellers_dataset",
    "olist_geolocation_dataset",
    "product_category_name_translation"
}

print("Approved tables:", len(APPROVED_TABLES))
print(sorted(APPROVED_TABLES))

Approved tables: 9
['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_orders_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation']


In [ ]:
def safe_run_sql(query, max_rows=1000, timeout_ms=3000):
    valid, message = validate_sql(query)

    if not valid:
        raise ValueError(f"Query rejected: {message}")

    conn = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True,
        timeout=3
    )

    start_time = time.perf_counter()

    def authorize(action, arg1, arg2, db_name, trigger):
        if action == sqlite3.SQLITE_READ:
            if arg1 and arg1 not in APPROVED_TABLES:
                return sqlite3.SQLITE_DENY
            return sqlite3.SQLITE_OK

        if action in {
            sqlite3.SQLITE_SELECT,
            sqlite3.SQLITE_FUNCTION,
            sqlite3.SQLITE_RECURSIVE
        }:
            return sqlite3.SQLITE_OK

        return sqlite3.SQLITE_DENY

    try:
        conn.execute("PRAGMA query_only = ON")
        conn.set_authorizer(authorize)

        def check_timeout():
            return int(
                time.perf_counter() - start_time > timeout_ms / 1000
            )

        conn.set_progress_handler(check_timeout, 1000)

        cursor = conn.execute(query)

        if cursor.description is None:
            raise ValueError("Query must return results.")

        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchmany(max_rows)

        return pd.DataFrame(rows, columns=columns)

    finally:
        conn.close()

In [ ]:
result = safe_run_sql("""
SELECT order_status, COUNT(*) AS total_orders
FROM olist_orders_dataset
GROUP BY order_status
ORDER BY total_orders DESC
""")

display(result)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [ ]:
tests = {
    "Valid query": """
        SELECT COUNT(*) AS total_orders
        FROM olist_orders_dataset
    """,
    "Write query": """
        DELETE FROM olist_orders_dataset
    """,
    "Unapproved table": """
        SELECT * FROM sqlite_master
    """,
    "Multiple statements": """
        SELECT 1; DROP TABLE olist_orders_dataset
    """
}

for name, query in tests.items():
    print(f"\n--- {name} ---")

    try:
        result = safe_run_sql(query)
        print("ALLOWED")
        display(result)
    except Exception as e:
        print("BLOCKED:", type(e).__name__, str(e))


--- Valid query ---
ALLOWED


,total_orders
0,99441



--- Write query ---
BLOCKED: ValueError Query rejected: Only SELECT queries are allowed

--- Unapproved table ---
BLOCKED: DatabaseError access to sqlite_master.type is prohibited

--- Multiple statements ---
BLOCKED: ValueError Query rejected: Multiple SQL statements are not allowed


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("No GPU available. CPU mode detected.")

PyTorch: 2.11.0+cpu
CUDA available: False
No GPU available. CPU mode detected.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("GPU not connected")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.6 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("Device:", model.device)
print("GPU:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda:0
GPU: Tesla T4


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an AI Business Analyst. Give clear, concise answers."
    },
    {
        "role": "user",
        "content": "What are 3 useful business insights a company can get from its order data?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

1. **Customer Purchase Patterns**: By analyzing order data, companies can identify which products or services are most frequently purchased together (co-purchases) and which are typically bought sequentially (sequence of purchases). This insight helps in optimizing product placement and cross-selling strategies.

2. **Seasonal Trends**: Order data can reveal patterns that indicate seasonal buying behaviors. For instance, a sudden spike in orders for winter clothing during the summer months suggests a need to stock up on these items early. Understanding these trends allows companies to better manage inventory and marketing efforts.

3. **Customer Lifetime Value (CLV)**: Analyzing order frequency and total spend over time can help determine customer lifetime value. Insights into how often customers make repeat purchases and the average amount spent per


In [ ]:
def generate_sql_with_llm(question):
    schema_text = "\n".join(
        f"{table}: " + ", ".join(
            column["column"] for column in columns
        )
        for table, columns in schema.items()
    )

    messages = [
        {
            "role": "system",
            "content": f"""
You are a SQLite SQL generation assistant.

Database schema:
{schema_text}

Rules:
- Generate exactly one read-only SELECT query.
- Use only tables and columns in the schema.
- Never generate INSERT, UPDATE, DELETE, DROP, or other modifications.
- Do not invent tables or columns.
- Return only SQL, without markdown fences or explanations.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    sql = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    sql = sql.replace("```sql", "").replace("```", "").strip()

    return sql

In [ ]:
question = "How many orders are there in the database?"

generated_sql = generate_sql_with_llm(question)

print("Question:", question)
print("\nGenerated SQL:\n", generated_sql)

NameError: name 'schema' is not defined

In [ ]:
schema = get_schema()

print("Tables loaded:", len(schema))
print("Sample tables:", list(schema.keys())[:5])

NameError: name 'get_schema' is not defined

In [ ]:
import sqlite3

DB_PATH = "/content/olist_business.db"

def get_schema():
    connection = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True
    )

    try:
        tables = connection.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
        """).fetchall()

        schema_info = {}

        for (table_name,) in tables:
            columns = connection.execute(
                f'PRAGMA table_info("{table_name}")'
            ).fetchall()

            schema_info[table_name] = [
                {"column": col[1], "type": col[2]}
                for col in columns
            ]

        return schema_info

    finally:
        connection.close()

schema = get_schema()

print("Tables loaded:", len(schema))
print("Sample tables:", list(schema.keys())[:5])

OperationalError: unable to open database file

In [ ]:

import os

print("Database exists:", os.path.exists("/content/olist_business.db"))
print("Current folder files:", os.listdir("/content")[:30])

Database exists: False
Current folder files: ['.config', 'sample_data']


In [ ]:
import os
import glob

print("CSV files in /content:")
print(glob.glob("/content/**/*.csv", recursive=True)[:20])

print("\nDataFrames variable exists:", "dataframes" in globals())

if "dataframes" in globals():
    print("DataFrames loaded:", len(dataframes))
    print("Sample names:", list(dataframes.keys())[:5])

CSV files in /content:
['/content/sample_data/mnist_train_small.csv', '/content/sample_data/mnist_test.csv', '/content/sample_data/california_housing_test.csv', '/content/sample_data/california_housing_train.csv']

DataFrames variable exists: False


In [ ]:
!pip install -q kagglehub

import kagglehub
import os

path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

print("Dataset path:", path)
print("CSV files:", len([
    f for f in os.listdir(path)
    if f.endswith(".csv")
]))
print(os.listdir(path))

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset path: /kaggle/input/brazilian-ecommerce
CSV files: 9
['olist_customers_dataset.csv', 'olist_sellers_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_order_items_dataset.csv', 'olist_products_dataset.csv', 'olist_geolocation_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_payments_dataset.csv']


In [ ]:
import pandas as pd
import os
import glob

csv_files = glob.glob(os.path.join(path, "*.csv"))

dataframes = {}

for file in csv_files:
    name = os.path.basename(file).replace(".csv", "")
    dataframes[name] = pd.read_csv(file)
    print(name, dataframes[name].shape)

print("\nTotal DataFrames:", len(dataframes))

olist_customers_dataset (99441, 5)
olist_sellers_dataset (3095, 4)
olist_order_reviews_dataset (99224, 7)
olist_order_items_dataset (112650, 7)
olist_products_dataset (32951, 9)
olist_geolocation_dataset (1000163, 5)
product_category_name_translation (71, 2)
olist_orders_dataset (99441, 8)
olist_order_payments_dataset (103886, 5)

Total DataFrames: 9


In [ ]:
import sqlite3
import os

DB_PATH = "/content/olist_business.db"

# Purana database recreate karo
conn = sqlite3.connect(DB_PATH)

for name, dataframe in dataframes.items():
    dataframe.to_sql(
        name,
        conn,
        if_exists="replace",
        index=False
    )

conn.close()

print("Database recreated:", os.path.exists(DB_PATH))
print("Tables created:", len(dataframes))

Database recreated: True
Tables created: 9


In [ ]:
import sqlite3

DB_PATH = "/content/olist_business.db"

def get_schema():
    connection = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True
    )

    try:
        tables = connection.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
        """).fetchall()

        schema_info = {}

        for (table_name,) in tables:
            columns = connection.execute(
                f'PRAGMA table_info("{table_name}")'
            ).fetchall()

            schema_info[table_name] = [
                {"column": col[1], "type": col[2]}
                for col in columns
            ]

        return schema_info

    finally:
        connection.close()

schema = get_schema()

print("Tables loaded:", len(schema))
print("Sample tables:", list(schema.keys())[:5])
print("Orders columns:",
      [c["column"] for c in schema["olist_orders_dataset"]])

Tables loaded: 9
Sample tables: ['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset']
Orders columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [ ]:
print("Model available:", "model" in globals())
print("Tokenizer available:", "tokenizer" in globals())
print("SQL generator available:", "generate_sql_with_llm" in globals())

Model available: True
Tokenizer available: True
SQL generator available: True


In [ ]:
question = "How many orders are there in the database?"

generated_sql = generate_sql_with_llm(question)

print("Question:", question)
print("\nGenerated SQL:\n", generated_sql)

Question: How many orders are there in the database?

Generated SQL:
 SELECT COUNT(*) AS order_count FROM olist_orders_dataset;


In [ ]:
valid, message = validate_sql(generated_sql)

print("Validation:", message)

if valid:
    try:
        result = safe_run_sql(generated_sql)
        display(result)
    except Exception as e:
        print("Execution error:", e)
else:
    print("Query rejected. Not executed.")

NameError: name 'validate_sql' is not defined

In [ ]:
import re

def validate_sql(query):
    if not isinstance(query, str) or not query.strip():
        return False, "Empty query"

    q = query.strip()

    if not re.match(r"(?is)^SELECT\b", q):
        return False, "Only SELECT queries are allowed"

    cleaned = q.rstrip().rstrip(";")

    if ";" in cleaned:
        return False, "Multiple SQL statements are not allowed"

    forbidden = (
        r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|ATTACH|"
        r"DETACH|REPLACE|PRAGMA|VACUUM|REINDEX|TRIGGER)\b"
    )

    if re.search(forbidden, q, re.IGNORECASE):
        return False, "Forbidden SQL keyword detected"

    return True, "Query passed basic checks"

print("SQL validator restored!")

SQL validator restored!


In [ ]:
valid, message = validate_sql(generated_sql)

print("Validation:", message)

if valid:
    try:
        result = safe_run_sql(generated_sql)
        display(result)
    except Exception as e:
        print("Execution error:", e)
else:
    print("Query rejected. Not executed.")

Validation: Query passed basic checks
Execution error: name 'safe_run_sql' is not defined


In [ ]:
import sqlite3
import time
import pandas as pd

DB_PATH = "/content/olist_business.db"

APPROVED_TABLES = {
    "olist_orders_dataset",
    "olist_customers_dataset",
    "olist_order_items_dataset",
    "olist_products_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_sellers_dataset",
    "olist_geolocation_dataset",
    "product_category_name_translation"
}

def safe_run_sql(query, max_rows=1000, timeout_ms=3000):
    valid, message = validate_sql(query)

    if not valid:
        raise ValueError(f"Query rejected: {message}")

    conn = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True,
        timeout=3
    )

    start_time = time.perf_counter()

    def authorize(action, arg1, arg2, db_name, trigger):
        if action == sqlite3.SQLITE_READ:
            if arg1 and arg1 not in APPROVED_TABLES:
                return sqlite3.SQLITE_DENY
            return sqlite3.SQLITE_OK

        allowed_actions = {
            sqlite3.SQLITE_SELECT,
            sqlite3.SQLITE_FUNCTION,
        }

        if hasattr(sqlite3, "SQLITE_RECURSIVE"):
            allowed_actions.add(sqlite3.SQLITE_RECURSIVE)

        if action in allowed_actions:
            return sqlite3.SQLITE_OK

        return sqlite3.SQLITE_DENY

    try:
        conn.execute("PRAGMA query_only = ON")
        conn.set_authorizer(authorize)

        def check_timeout():
            return int(
                time.perf_counter() - start_time
                > timeout_ms / 1000
            )

        conn.set_progress_handler(check_timeout, 1000)

        cursor = conn.execute(query)

        if cursor.description is None:
            raise ValueError("Query must return results.")

        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchmany(max_rows)

        return pd.DataFrame(rows, columns=columns)

    finally:
        conn.close()

print("Safe SQL runner restored!")

Safe SQL runner restored!


In [ ]:
valid, message = validate_sql(generated_sql)

print("Validation:", message)

if valid:
    try:
        result = safe_run_sql(generated_sql)
        display(result)
    except Exception as e:
        print("Execution error:", e)
else:
    print("Query rejected. Not executed.")

Validation: Query passed basic checks


,order_count
0,99441


In [ ]:
test_questions = [
    "How many orders are there in each order status?",
    "Show the top 10 customers by number of orders."
]

for question in test_questions:
    print("\nQuestion:", question)

    try:
        sql = generate_sql_with_llm(question)
        print("Generated SQL:", sql)

        valid, message = validate_sql(sql)
        print("Validation:", message)

        if valid:
            result = safe_run_sql(sql)
            display(result)
        else:
            print("Query rejected.")

    except Exception as e:
        print("Error:", e)


Question: How many orders are there in each order status?
Generated SQL: SELECT order_status, COUNT(*) AS number_of_orders FROM olist_orders_dataset GROUP BY order_status
Validation: Query passed basic checks


,order_status,number_of_orders
0,approved,2
1,canceled,625
2,created,5
3,delivered,96478
4,invoiced,314
5,processing,301
6,shipped,1107
7,unavailable,609



Question: Show the top 10 customers by number of orders.
Generated SQL: SELECT customer_unique_id, COUNT(order_id) AS order_count FROM olist_orders_dataset GROUP BY customer_unique_id ORDER BY order_count DESC LIMIT 10
Validation: Query passed basic checks
Error: no such column: customer_unique_id


In [ ]:
def generate_sql_with_llm(question):
    schema_text = "\n".join(
        f"{table}: " + ", ".join(
            column["column"] for column in columns
        )
        for table, columns in schema.items()
    )

    messages = [
        {
            "role": "system",
            "content": f"""
You are an expert SQLite query generator.

DATABASE SCHEMA:
{schema_text}

RULES:
1. Generate exactly one read-only SELECT query.
2. Use only tables and columns explicitly listed in the schema.
3. Never guess a column name.
4. For customer questions, inspect the schema carefully.
5. The orders table uses customer_id to link to the
   customers table, where customer_unique_id exists.
6. Use JOIN when information requires multiple tables.
7. Return SQL only, without markdown or explanations.

8. Generate SQLite-compatible SQL only.
9. Never use YEAR(), MONTH(), or DATEPART().
10. For year filtering, use strftime('%Y', date_column) = '2018'.

"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    sql = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return sql.replace("```sql", "").replace("```", "").strip()

print("Improved SQL generator ready!")

Improved SQL generator ready!


In [ ]:
question = "Show the top 10 customers by number of orders."

sql = generate_sql_with_llm(question)

print("Generated SQL:\n", sql)

valid, message = validate_sql(sql)
print("\nValidation:", message)

if valid:
    try:
        display(safe_run_sql(sql))
    except Exception as e:
        print("Execution error:", e)

Generated SQL:
 SELECT c.customer_unique_id, COUNT(o.order_id) AS order_count
FROM olist_customers_dataset c
JOIN olist_orders_dataset o ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
ORDER BY order_count DESC
LIMIT 10

Validation: Query passed basic checks


,customer_unique_id,order_count
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6


In [ ]:
def business_copilot_query(question, max_retries=1):
    last_error = None

    for attempt in range(max_retries + 1):
        if attempt == 0:
            sql = generate_sql_with_llm(question)
        else:
            schema_text = "\n".join(
                f"{table}: " + ", ".join(
                    col["column"] for col in columns
                )
                for table, columns in schema.items()
            )

            messages = [
                {
                    "role": "system",
                    "content": f"""
Generate one SQLite SELECT query only.
Use only the tables and columns in this schema:
{schema_text}
Return SQL only. No markdown.
"""
                },
                {
                    "role": "user",
                    "content": f"""
Original question: {question}
Previous SQL failed with this error: {last_error}
Generate a corrected query. Verify every table and column.
"""
                }
            ]

            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(
                prompt, return_tensors="pt"
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=200,
                    do_sample=False
                )

            sql = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            ).replace("```sql", "").replace("```", "").strip()

        valid, message = validate_sql(sql)

        print(f"\nAttempt {attempt + 1}")
        print("SQL:", sql)
        print("Validation:", message)

        if not valid:
            last_error = message
            continue

        try:
            result = safe_run_sql(sql)
            print("Query successful!")
            return {"sql": sql, "result": result, "error": None}
        except Exception as e:
            last_error = str(e)
            print("Query failed:", last_error)

    return {"sql": None, "result": None, "error": last_error}

In [ ]:
response = business_copilot_query(
    "Show the top 10 customers by number of orders."
)

if response["result"] is not None:
    display(response["result"])
else:
    print("Final error:", response["error"])


Attempt 1
SQL: SELECT c.customer_unique_id, COUNT(o.order_id) AS order_count
FROM olist_customers_dataset c
JOIN olist_orders_dataset o ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
ORDER BY order_count DESC
LIMIT 10
Validation: Query passed basic checks
Query successful!


,customer_unique_id,order_count
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6


In [ ]:
def explain_business_result(question, result_df):
    if result_df is None or result_df.empty:
        return "No results were returned for this question."

    result_text = result_df.to_string(index=False, max_rows=20)

    messages = [
        {
            "role": "system",
            "content": """
You are a careful business analyst.
Explain the provided SQL results in simple language.
Use only facts supported by the results.
Do not invent causes, trends, revenue, or recommendations.
Clearly distinguish observations from suggestions.
"""
        },
        {
            "role": "user",
            "content": f"""
Business question:
{question}

SQL query result:
{result_text}

Explain the main findings in 3-5 concise bullet points.
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


question = "Show the top 10 customers by number of orders."

response = business_copilot_query(question)

if response["result"] is not None:
    explanation = explain_business_result(
        question,
        response["result"]
    )
    print(explanation)
else:
    print("Query failed:", response["error"])


Attempt 1
SQL: SELECT c.customer_unique_id, COUNT(o.order_id) AS order_count
FROM olist_customers_dataset c
JOIN olist_orders_dataset o ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
ORDER BY order_count DESC
LIMIT 10
Validation: Query passed basic checks
Query successful!
- The top 10 customers by number of orders are identified.
- The first customer (customer_unique_id: 8d50f5eadf50201ccdcedfb9e2ac8455) has placed 17 orders.
- Customers with 7 orders each include those with unique IDs: ca77025e7201e3b30c44b472ff346268, 6469f99c1f9dfae7733b25662e7f1782, and 1b6c7548a2a1f9037c1fd3ddfed95f33.
- Other customers have placed between 6 and 1 orders.


In [ ]:
benchmark = [
    {
        "question": "How many orders are there in the database?",
        "expected_sql_hint": "COUNT(*) from olist_orders_dataset"
    },
    {
        "question": "How many orders are there in each order status?",
        "expected_sql_hint": "GROUP BY order_status"
    },
    {
        "question": "Show the top 10 customers by number of orders.",
        "expected_sql_hint": "JOIN customers and orders, group by customer"
    },
    {
        "question": "How many customers are in the database?",
        "expected_sql_hint": "COUNT(*) from olist_customers_dataset"
    },
    {
        "question": "Show the 10 most common order statuses.",
        "expected_sql_hint": "GROUP BY order_status and order by count"
    }
]

print("Benchmark questions:", len(benchmark))

for i, item in enumerate(benchmark, 1):
    print(f"{i}. {item['question']}")

Benchmark questions: 5
1. How many orders are there in the database?
2. How many orders are there in each order status?
3. Show the top 10 customers by number of orders.
4. How many customers are in the database?
5. Show the 10 most common order statuses.


In [ ]:
benchmark_results = []

for item in benchmark:
    print("\nQuestion:", item["question"])

    response = business_copilot_query(item["question"])

    success = response["result"] is not None

    benchmark_results.append({
        "question": item["question"],
        "success": success,
        "error": response["error"]
    })

    print("Status:", "SUCCESS" if success else "FAILED")

import pandas as pd

benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)

print(
    "Execution success rate:",
    round(benchmark_df["success"].mean() * 100, 1),
    "%"
)


Question: How many orders are there in the database?

Attempt 1
SQL: SELECT COUNT(*) AS order_count FROM olist_orders_dataset
Validation: Query passed basic checks
Query successful!
Status: SUCCESS

Question: How many orders are there in each order status?

Attempt 1
SQL: SELECT order_status, COUNT(*) AS number_of_orders FROM olist_orders_dataset GROUP BY order_status
Validation: Query passed basic checks
Query successful!
Status: SUCCESS

Question: Show the top 10 customers by number of orders.

Attempt 1
SQL: SELECT c.customer_unique_id, COUNT(o.order_id) AS order_count
FROM olist_customers_dataset c
JOIN olist_orders_dataset o ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
ORDER BY order_count DESC
LIMIT 10
Validation: Query passed basic checks
Query successful!
Status: SUCCESS

Question: How many customers are in the database?

Attempt 1
SQL: SELECT COUNT(DISTINCT customer_unique_id) AS number_of_customers FROM olist_customers_dataset
Validation: Query passed basic

,question,success,error
0,How many orders are there in the database?,True,None
1,How many orders are there in each order status?,True,None
2,Show the top 10 customers by number of orders.,True,None
3,How many customers are in the database?,True,None
4,Show the 10 most common order statuses.,True,None


Execution success rate: 100.0 %


In [ ]:

orders_df = dataframes["olist_orders_dataset"]
customers_df = dataframes["olist_customers_dataset"]

expected_orders = len(orders_df)
expected_status_counts = (
    orders_df["order_status"].value_counts().to_dict()
)
expected_customers = customers_df["customer_unique_id"].nunique()

print("Expected total orders:", expected_orders)
print("Expected unique customers:", expected_customers)
print("\nExpected order counts by status:")
print(expected_status_counts)

Expected total orders: 99441
Expected unique customers: 96096

Expected order counts by status:
{'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}


In [ ]:

sql_orders = safe_run_sql("""
    SELECT COUNT(*) AS total_orders
    FROM olist_orders_dataset
""").iloc[0, 0]

pandas_orders = len(dataframes["olist_orders_dataset"])

print("Total orders - SQL:", sql_orders)
print("Total orders - Pandas:", pandas_orders)
print("Match:", sql_orders == pandas_orders)



sql_customers = safe_run_sql("""
    SELECT COUNT(DISTINCT customer_unique_id) AS total_customers
    FROM olist_customers_dataset
""").iloc[0, 0]

pandas_customers = (
    dataframes["olist_customers_dataset"]["customer_unique_id"].nunique()
)

print("\nUnique customers - SQL:", sql_customers)
print("Unique customers - Pandas:", pandas_customers)
print("Match:", sql_customers == pandas_customers)



sql_status = safe_run_sql("""
    SELECT order_status, COUNT(*) AS total
    FROM olist_orders_dataset
    GROUP BY order_status
""")

pandas_status = (
    dataframes["olist_orders_dataset"]["order_status"]
    .value_counts()
    .rename_axis("order_status")
    .reset_index(name="total")
)

sql_status_dict = dict(zip(sql_status["order_status"], sql_status["total"]))
pandas_status_dict = dict(zip(pandas_status["order_status"], pandas_status["total"]))

print("\nOrder status counts match:", sql_status_dict == pandas_status_dict)
print("SQL status counts:", sql_status_dict)

Total orders - SQL: 99441
Total orders - Pandas: 99441
Match: True

Unique customers - SQL: 96096
Unique customers - Pandas: 96096
Match: True

Order status counts match: True
SQL status counts: {'approved': 2, 'canceled': 625, 'created': 5, 'delivered': 96478, 'invoiced': 314, 'processing': 301, 'shipped': 1107, 'unavailable': 609}


In [ ]:



total_tests = len(benchmark)
passed = 0
failed_cases = []

for item in benchmark:
    question = item["question"]
    response = business_copilot_query(question)

    if response["error"] is None and response["result"] is not None:
        passed += 1
    else:
        failed_cases.append({
            "question": question,
            "error": response["error"]
        })

print("Total benchmark tests:", total_tests)
print("Successful executions:", passed)
print("Failed executions:", total_tests - passed)
print("Execution success rate:", round(passed / total_tests * 100, 2), "%")
print("Failed cases:", failed_cases)


Attempt 1
SQL: SELECT COUNT(*) AS order_count FROM olist_orders_dataset
Validation: Query passed basic checks
Query successful!

Attempt 1
SQL: SELECT order_status, COUNT(*) AS number_of_orders FROM olist_orders_dataset GROUP BY order_status
Validation: Query passed basic checks
Query successful!

Attempt 1
SQL: SELECT c.customer_unique_id, COUNT(o.order_id) AS order_count
FROM olist_customers_dataset c
JOIN olist_orders_dataset o ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
ORDER BY order_count DESC
LIMIT 10
Validation: Query passed basic checks
Query successful!

Attempt 1
SQL: SELECT COUNT(DISTINCT customer_unique_id) AS number_of_customers FROM olist_customers_dataset
Validation: Query passed basic checks
Query successful!

Attempt 1
SQL: SELECT order_status, COUNT(*) AS status_count
FROM olist_orders_dataset
GROUP BY order_status
ORDER BY status_count DESC
LIMIT 10
Validation: Query passed basic checks
Query successful!
Total benchmark tests: 5
Successful execut

In [ ]:



tests = [
    {
        "question": "How many orders are there in the database?",
        "expected": 99441
    },
    {
        "question": "How many customers are in the database?",
        "expected": 96096
    }
]

for test in tests:
    response = business_copilot_query(test["question"])

    print("\nQuestion:", test["question"])

    if response["error"] is not None:
        print("ERROR:", response["error"])
        continue

    result = response["result"]
    actual = result.iloc[0, 0]

    print("Model result:", actual)
    print("Expected result:", test["expected"])
    print("Correct:", actual == test["expected"])


Attempt 1
SQL: SELECT COUNT(*) AS order_count FROM olist_orders_dataset
Validation: Query passed basic checks
Query successful!

Question: How many orders are there in the database?
Model result: 99441
Expected result: 99441
Correct: True

Attempt 1
SQL: SELECT COUNT(DISTINCT customer_unique_id) AS number_of_customers FROM olist_customers_dataset
Validation: Query passed basic checks
Query successful!

Question: How many customers are in the database?
Model result: 96096
Expected result: 96096
Correct: True


In [ ]:

stress_tests = [
    "Show the top 5 product categories by number of products.",
    "What is the average payment value?",
    "How many orders are currently delivered?",
    "Show the 5 states with the most customers.",
    "How many orders were placed in 2018?"
]

for question in stress_tests:
    print("\n" + "=" * 60)
    print("QUESTION:", question)

    response = business_copilot_query(question)

    if response["error"] is not None:
        print("STATUS: FAILED")
        print("ERROR:", response["error"])
    else:
        print("STATUS: EXECUTED")
        print("GENERATED SQL:\n", response["sql"])
        print("RESULT:\n", response["result"].head(5))


QUESTION: Show the top 5 product categories by number of products.

Attempt 1
SQL: SELECT product_category_name_english, COUNT(*) AS product_count
FROM olist_products_dataset
GROUP BY product_category_name_english
ORDER BY product_count DESC
LIMIT 5
Validation: Query passed basic checks
Query failed: no such column: product_category_name_english

Attempt 2
SQL: SELECT product_category_name_english, COUNT(*) AS category_count
FROM olist_products_dataset
JOIN product_category_name_translation
ON olist_products_dataset.product_category_name = product_category_name_translation.product_category_name
GROUP BY product_category_name_english
ORDER BY category_count DESC
LIMIT 5
Validation: Query passed basic checks
Query successful!
STATUS: EXECUTED
GENERATED SQL:
 SELECT product_category_name_english, COUNT(*) AS category_count
FROM olist_products_dataset
JOIN product_category_name_translation
ON olist_products_dataset.product_category_name = product_category_name_translation.product_category

In [ ]:


avg_payment = safe_run_sql("""
    SELECT AVG(payment_value) AS average_payment_value
    FROM olist_order_payments_dataset
""")
print("Average payment:")
print(avg_payment)

orders_2018 = safe_run_sql("""
    SELECT COUNT(*) AS order_count_2018
    FROM olist_orders_dataset
    WHERE strftime('%Y', order_purchase_timestamp) = '2018'
""")
print("\nOrders placed in 2018:")
print(orders_2018)

Average payment:
   average_payment_value
0              154.10038

Orders placed in 2018:
   order_count_2018
0             54011


In [ ]:

question = "What is the average payment value?"

print("1. DIRECT SQL RESULT")
direct_result = safe_run_sql("""
    SELECT AVG(payment_value) AS average_payment_value
    FROM olist_order_payments_dataset
""")
print(direct_result)

print("\n2. COPILOT RESULT")
copilot_result = business_copilot_query(question)
print("SQL:", copilot_result["sql"])
print("Error:", copilot_result["error"])
print("Result:")
print(copilot_result["result"])

1. DIRECT SQL RESULT
   average_payment_value
0              154.10038

2. COPILOT RESULT

Attempt 1
SQL: SELECT AVG(payment_value) AS average_payment_value FROM olist_order_payments_dataset
Validation: Query passed basic checks
Query successful!
SQL: SELECT AVG(payment_value) AS average_payment_value FROM olist_order_payments_dataset
Error: None
Result:
   average_payment_value
0              154.10038


In [ ]:

question = "How many orders were placed in 2018?"

response = business_copilot_query(question)

print("SQL:", response["sql"])
print("Error:", response["error"])
print("Result:")
print(response["result"])


Attempt 1
SQL: SELECT COUNT(*) AS order_count_2018 FROM olist_orders_dataset WHERE YEAR(order_purchase_timestamp) = 2018
Validation: Query passed basic checks
Query failed: no such function: YEAR

Attempt 2
SQL: SELECT COUNT(*) AS order_count
FROM olist_orders_dataset
WHERE YEAR(order_purchase_timestamp) = 2018;
Validation: Query passed basic checks
Query failed: no such function: YEAR
SQL: None
Error: no such function: YEAR
Result:
None


In [ ]:

sql = generate_sql_with_llm(
    "How many orders were placed in 2018?"
)
print(sql)
print(safe_run_sql(sql))

SELECT COUNT(*) AS order_count_2018 FROM olist_orders_dataset WHERE strftime('%Y', order_purchase_timestamp) = '2018'
   order_count_2018
0             54011


In [ ]:
remaining_tests = [
    "How many orders are currently delivered?",
    "Show the 5 states with the most customers."
]

for question in remaining_tests:
    print("\n" + "=" * 50)
    print("QUESTION:", question)

    response = business_copilot_query(question)

    print("SQL:", response["sql"])
    print("ERROR:", response["error"])

    if response["result"] is not None:
        print(response["result"].head(5))


QUESTION: How many orders are currently delivered?

Attempt 1
SQL: SELECT COUNT(*) AS num_delivered_orders FROM olist_orders_dataset WHERE order_delivered_customer_date IS NOT NULL AND order_delivered_customer_date > CURRENT_DATE
Validation: Query passed basic checks
Query successful!
SQL: SELECT COUNT(*) AS num_delivered_orders FROM olist_orders_dataset WHERE order_delivered_customer_date IS NOT NULL AND order_delivered_customer_date > CURRENT_DATE
ERROR: None
   num_delivered_orders
0                     0

QUESTION: Show the 5 states with the most customers.

Attempt 1
SQL: SELECT seller_state, COUNT(*) AS customer_count
FROM olist_sellers_dataset
GROUP BY seller_state
ORDER BY customer_count DESC
LIMIT 5;
Validation: Query passed basic checks
Query successful!
SQL: SELECT seller_state, COUNT(*) AS customer_count
FROM olist_sellers_dataset
GROUP BY seller_state
ORDER BY customer_count DESC
LIMIT 5;
ERROR: None
  seller_state  customer_count
0           SP            1849
1         

In [ ]:


delivered = safe_run_sql("""
    SELECT COUNT(*) AS delivered_orders
    FROM olist_orders_dataset
    WHERE order_status = 'delivered'
""")

print("Delivered orders:")
print(delivered)


customers_by_state = safe_run_sql("""
    SELECT customer_state, COUNT(*) AS customer_count
    FROM olist_customers_dataset
    GROUP BY customer_state
    ORDER BY customer_count DESC
    LIMIT 5
""")

print("\nTop 5 states by customer count:")
print(customers_by_state)

Delivered orders:
   delivered_orders
0             96478

Top 5 states by customer count:
  customer_state  customer_count
0             SP           41746
1             RJ           12852
2             MG           11635
3             RS            5466
4             PR            5045


In [ ]:


evaluation_report = [
    {
        "test": "Total orders",
        "expected": 99441,
        "actual": safe_run_sql(
            "SELECT COUNT(*) FROM olist_orders_dataset"
        ).iloc[0, 0]
    },
    {
        "test": "Unique customers",
        "expected": 96096,
        "actual": safe_run_sql(
            "SELECT COUNT(DISTINCT customer_unique_id) "
            "FROM olist_customers_dataset"
        ).iloc[0, 0]
    },
    {
        "test": "Delivered orders",
        "expected": 96478,
        "actual": safe_run_sql(
            "SELECT COUNT(*) FROM olist_orders_dataset "
            "WHERE order_status = 'delivered'"
        ).iloc[0, 0]
    },
    {
        "test": "Orders in 2018",
        "expected": 54011,
        "actual": safe_run_sql(
            "SELECT COUNT(*) FROM olist_orders_dataset "
            "WHERE strftime('%Y', order_purchase_timestamp) = '2018'"
        ).iloc[0, 0]
    }
]

report = pd.DataFrame(evaluation_report)
report["correct"] = report["expected"] == report["actual"]

display(report)
print("\nVerified accuracy:", round(report["correct"].mean() * 100, 2), "%")

,test,expected,actual,correct
0,Total orders,99441,99441,True
1,Unique customers,96096,96096,True
2,Delivered orders,96478,96478,True
3,Orders in 2018,54011,54011,True



Verified accuracy: 100.0 %


In [ ]:

!pip -q install gradio

In [ ]:
print("Model:", "model" in globals())
print("Tokenizer:", "tokenizer" in globals())
print("Copilot function:", "business_copilot_query" in globals())
print("Database:", __import__("os").path.exists("/content/olist_business.db"))

Model: True
Tokenizer: True
Copilot function: True
Database: True


In [ ]:

import gradio as gr
import pandas as pd

custom_css = """
.gradio-container {
    max-width: 1450px !important;
    margin: auto !important;
    font-family: Inter, ui-sans-serif, system-ui, sans-serif !important;
}
#app-header {
    border-radius: 18px;
    padding: 25px 28px;
    background: linear-gradient(120deg, #101b33, #1d3f78);
    color: white;
    margin-bottom: 18px;
}
#app-header h1 { color: white !important; margin-bottom: 8px; }
#app-header p { color: #cfdbf3 !important; }
#sidebar {
    border: 1px solid #dfe6f1;
    border-radius: 16px;
    padding: 18px;
}
#main-panel {
    border: 1px solid #dfe6f1;
    border-radius: 16px;
    padding: 20px;
}
#run-button {
    background: #2863d7 !important;
    color: white !important;
    border: none !important;
}
#run-button:hover { background: #194eaf !important; }
#status-box textarea { font-weight: 600 !important; }
"""

def run_copilot_ui(question):
    if not question or not question.strip():
        return "Please enter a question.", "", pd.DataFrame()

    try:
        response = business_copilot_query(question.strip())
        sql = response.get("sql") or ""
        result = response.get("result")
        error = response.get("error")

        if error:
            return f"Query failed: {error}", sql, pd.DataFrame()

        if result is None:
            return "No results returned.", sql, pd.DataFrame()

        return f"✓ Success · {len(result)} row(s) returned", sql, result

    except Exception as e:
        return f"Unexpected error: {e}", "", pd.DataFrame()


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
        neutral_hue="slate"
    ),
    css=custom_css,
    title="AI Business Copilot Pro"
) as demo:

    gr.HTML("""
    <div id="app-header">
      <div style="font-size:12px;letter-spacing:2px;font-weight:700">
        AI BUSINESS INTELLIGENCE
      </div>
      <h1>Business Copilot</h1>
      <p>Turn natural-language questions into useful business insights.</p>
    </div>
    """)

    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=220):
            with gr.Group(elem_id="sidebar"):
                gr.Markdown("## Workspace")
                gr.Markdown(
                    "🔵 **AI Analyst**\n\n"
                    "🗄️ Data Explorer\n\n"
                    "📋 SQL Inspector\n\n"
                    "---\n\n"
                    "### Connected source\n"
                    "**Olist Ecommerce**\n\n"
                    "`SQLite · Read-only`\n\n"
                    "### Model\n"
                    "`Qwen 2.5 3B`"
                )

        with gr.Column(scale=4, min_width=320):
            with gr.Group(elem_id="main-panel"):
                gr.Markdown("## Ask your data")
                gr.Markdown(
                    "Write a question in plain English. "
                    "Review the generated SQL and returned results."
                )

                question = gr.Textbox(
                    label="Business question",
                    placeholder="e.g. How many orders were delivered?",
                    lines=2
                )

                gr.Examples(
                    examples=[
                        ["How many orders are there in the database?"],
                        ["How many orders were delivered?"],
                        ["What is the average payment value?"],
                        ["Show the 5 states with the most customers."]
                    ],
                    inputs=question
                )

                run_button = gr.Button(
                    "✦ Generate insight",
                    variant="primary",
                    elem_id="run-button"
                )

            status = gr.Textbox(
                label="Execution status",
                interactive=False,
                elem_id="status-box"
            )

            with gr.Group(elem_id="main-panel"):
                gr.Markdown("## SQL Inspector")
                sql_output = gr.Code(
                    label="Generated SQL",
                    language="sql"
                )

            with gr.Group(elem_id="main-panel"):
                gr.Markdown("## Query results")
                result_output = gr.Dataframe(
                    label="Returned data",
                    interactive=False
                )

            run_button.click(
                fn=run_copilot_ui,
                inputs=question,
                outputs=[status, sql_output, result_output]
            )

demo.launch(share=True, debug=True)